# Sample Growth Analysis with pipelinesam (UNet 0.7)

This notebook demonstrates automated sample growth analysis using the `pipelinesam` package and the UNet model (weight 0.7).
- Images are loaded recursively from the `input_images/` directory, including nested folders.
- If `input_images/image_metadata.csv` exists, it is used as a base and merged with any additional images found on disk.
- Results and overlays are saved to `outputs/pipelinesam_notebook/`.

**Note:** This notebook is intended for reproducible, automated analysis of sample growth.

In [2]:
import os
import sys
import math
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from PIL import Image
from scipy import ndimage
from skimage import exposure, filters, measure, morphology
from skimage.filters import frangi, meijering, threshold_local
from skimage.measure import shannon_entropy
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=r'Parameter `min_size` is deprecated.*', category=FutureWarning)

CWD = Path.cwd().resolve()
# In repo: detect root by locating the model checkpoint.
# Post pip-install: ROOT falls back to CWD; set UNET_MODEL to override.
ROOT = next(
    (c for c in (CWD, CWD.parent) if (c / 'models' / 'best_area_w_0.7.pt').exists()),
    CWD,
)

INPUT_IMAGES_DIR = ROOT / 'input_images'
OUTPUTS_DIR = ROOT / 'outputs' / 'pipelinesam_notebook'
OVERLAY_DIR = OUTPUTS_DIR / 'overlays'
for folder in (OUTPUTS_DIR, OVERLAY_DIR):
    folder.mkdir(parents=True, exist_ok=True)

os.environ.setdefault('IMG_DIR', str(INPUT_IMAGES_DIR))
os.environ.setdefault('UNET_MODEL', str(ROOT / 'models' / 'best_area_w_0.7.pt'))

import pipelinesam.pipeline as sam_pipeline

DEVICE = sam_pipeline.DEVICE
MODEL_PATH = sam_pipeline.MODEL_PATH

print('Notebook root =', ROOT)
print('Input images dir =', INPUT_IMAGES_DIR)
print('Outputs dir =', OUTPUTS_DIR)
print('Device =', DEVICE)
print('UNet model path =', MODEL_PATH)
print('input_images exists =', INPUT_IMAGES_DIR.exists())


Notebook root = /Users/wot25kir/magnaporthe-growth-analyzer
Input images dir = /Users/wot25kir/magnaporthe-growth-analyzer/input_images
Outputs dir = /Users/wot25kir/magnaporthe-growth-analyzer/outputs/pipelinesam_notebook
Device = mps
UNet model path = /Users/wot25kir/magnaporthe-growth-analyzer/models/best_area_w_0.7.pt
input_images exists = True


In [3]:
IMAGE_EXTS = {
    '.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.webp', '.gif',
    '.heic', '.heif', '.dng', '.cr2', '.nef', '.arw', '.raf', '.orf', '.rw2',
}


def get_image_date(path: Path):
    import re
    from datetime import datetime

    m = re.search(r'(\d{8})', path.stem)
    if m:
        try:
            return datetime.strptime(m.group(1), '%Y%m%d')
        except Exception:
            pass
    try:
        exif = Image.open(path)._getexif()
        if exif and 36867 in exif:
            return datetime.strptime(exif[36867], '%Y:%m:%d %H:%M:%S')
    except Exception:
        pass
    return datetime.fromtimestamp(path.stat().st_mtime)


def calc_day_code(exp, img):
    from datetime import datetime

    try:
        e = datetime.strptime(exp, '%Y-%m-%d')
        return f'd{max(1, (img.date() - e.date()).days + 1):02d}'
    except Exception:
        return ''


def discover_images(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(p for p in root.rglob('*') if p.suffix.lower() in IMAGE_EXTS)


def _build_record(path: Path, experiment_name: str, experiment_date, image_date, day_code: str) -> dict:
    return {
        'image_path': path.relative_to(INPUT_IMAGES_DIR).as_posix(),
        'file_path': path,
        'file_name': path.name,
        'experiment_name': experiment_name,
        'experiment_date': experiment_date,
        'image_date': image_date,
        'day_code': day_code,
        'user_name': '',
        'containers_count': '',
    }


def infer_experiment_name(path: Path) -> str:
    rel = path.relative_to(INPUT_IMAGES_DIR)
    if len(rel.parts) > 1:
        return rel.parts[0]
    if '_' in path.stem:
        return path.stem.split('_')[0]
    return path.parent.name or 'experiment'


def load_or_build_manifest(images: list[Path]) -> pd.DataFrame:
    metadata_csv = INPUT_IMAGES_DIR / 'image_metadata.csv'
    if metadata_csv.exists():
        df = pd.read_csv(metadata_csv)
        if 'image_path' not in df.columns:
            return pd.DataFrame()
        df = df.copy()
        df['image_path'] = df['image_path'].astype(str)
        df['file_path'] = df['image_path'].map(lambda rel: (INPUT_IMAGES_DIR / rel).resolve())
        df = df[df['file_path'].map(Path.exists)].copy()
        if df.empty:
            return pd.DataFrame()
        if 'image_date' in df.columns:
            df['image_date'] = pd.to_datetime(df['image_date'], errors='coerce')
        if 'experiment_date' in df.columns:
            df['experiment_date'] = pd.to_datetime(df['experiment_date'], errors='coerce')
        if 'day_code' not in df.columns:
            df['day_code'] = ''
        if 'experiment_name' not in df.columns:
            df['experiment_name'] = df['file_path'].map(infer_experiment_name)

        seen = {Path(p).resolve() for p in df['file_path'].astype(str)}
        missing = [path for path in images if path.resolve() not in seen]
        if missing:
            grouped_missing: dict[str, list[Path]] = {}
            for path in missing:
                grouped_missing.setdefault(infer_experiment_name(path), []).append(path)
            extras = []
            for experiment_name, paths in grouped_missing.items():
                dated = [(path, get_image_date(path)) for path in paths]
                experiment_date = min(dt for _, dt in dated)
                for path, image_date in dated:
                    extras.append(
                        _build_record(
                            path,
                            experiment_name,
                            pd.to_datetime(experiment_date),
                            pd.to_datetime(image_date),
                            calc_day_code(pd.to_datetime(experiment_date).strftime('%Y-%m-%d'), image_date),
                        )
                    )
            if extras:
                df = pd.concat([df, pd.DataFrame(extras)], ignore_index=True, sort=False)
        return df

    records = []
    grouped: dict[str, list[Path]] = {}
    for path in images:
        grouped.setdefault(infer_experiment_name(path), []).append(path)
    for experiment_name, paths in grouped.items():
        dated = [(path, get_image_date(path)) for path in paths]
        experiment_date = min(dt for _, dt in dated)
        for path, image_date in dated:
            day_code = calc_day_code(experiment_date.strftime('%Y-%m-%d'), image_date)
            records.append(_build_record(path, experiment_name, experiment_date, image_date, day_code))
    df = pd.DataFrame(records)
    if not df.empty:
        df['image_date'] = pd.to_datetime(df['image_date'])
        df['experiment_date'] = pd.to_datetime(df['experiment_date'])
    return df


IMAGES = discover_images(INPUT_IMAGES_DIR)
MANIFEST = load_or_build_manifest(IMAGES)
print('Images found =', len(IMAGES))
print('Manifest rows =', len(MANIFEST))
if not MANIFEST.empty:
    display(MANIFEST.head())
else:
    display(Markdown('Put images under `input_images/` to run the notebook.'))


Images found = 3
Manifest rows = 3


,image_path,experiment_name,experiment_date,image_date,day_code,user_name,plates_count,remind_me,image,experiment,date,day,user,plates,remind,file_path
0,06FEB/20260210_P001_06-FEB_WT_PCBM_SUB_d04_TOP...,magnaporthe,2026-02-04,2026-02-10,d07,rex,1,NaN,06FEB/20260210_P001_06-FEB_WT_PCBM_SUB_d04_TOP...,magnaporthe,2026-02-04,d07,rex,1,NaN,/Users/wot25kir/magnaporthe-growth-analyzer/in...
1,06FEB/20260212_P001_06-FEB_WT_PCBM_SUB_d06_TOP...,magnaporthe,2026-02-05,2026-02-12,d08,rex,1,NaN,06FEB/20260212_P001_06-FEB_WT_PCBM_SUB_d06_TOP...,magnaporthe,2026-02-05,d08,rex,1,NaN,/Users/wot25kir/magnaporthe-growth-analyzer/in...
2,06FEB/20260216_P001_06-FEB_WT_PCBM_SUB_d10_TOP...,magnaporthe,2026-02-16,2026-02-16,d01,rex,1,NaN,06FEB/20260216_P001_06-FEB_WT_PCBM_SUB_d10_TOP...,magnaporthe,2026-02-16,d01,rex,1,NaN,/Users/wot25kir/magnaporthe-growth-analyzer/in...


In [ ]:
def _load_image_as_pil(path: Path) -> Image.Image:
    suffix = str(path.suffix).lower()
    if suffix in {'.heic', '.heif'}:
        try:
            import pillow_heif
            pillow_heif.register_heif_opener()
        except ImportError:
            pass
    if suffix in {'.dng', '.cr2', '.nef', '.arw', '.raf', '.orf', '.rw2'}:
        try:
            import rawpy
            with rawpy.imread(str(path)) as raw:
                return Image.fromarray(raw.postprocess())
        except Exception:
            pass
    return Image.open(path).convert('RGB')


MODEL = sam_pipeline.get_model()


def _overlay_mask(rgb: np.ndarray, mask: np.ndarray, color=(38, 166, 154), alpha: float = 0.35) -> np.ndarray:
    vis = rgb.copy().astype(np.float32)
    mask = np.asarray(mask, dtype=bool)
    vis[mask] = vis[mask] * (1.0 - alpha) + np.array(color, dtype=np.float32) * alpha
    return np.clip(vis, 0, 255).astype(np.uint8)


def trace_image(image_path: Path, meta: dict | None = None) -> dict[str, object]:
    image = _load_image_as_pil(image_path)
    rgb = np.asarray(image)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    height, width = gray.shape

    center_x, center_y, radius_px = sam_pipeline.detect_container(gray)
    container_mask = sam_pipeline.circle_mask(width, height, center_x, center_y, radius_px)
    pixel_to_mm = (sam_pipeline.CONTAINER_MM / 2.0) / max(radius_px, 1.0)

    # Classical mask kept only as a fallback if UNet produces nothing
    classical_mask = sam_pipeline.build_mask(gray, center_x, center_y, radius_px) & container_mask
    classical_mask = sam_pipeline.largest_component(classical_mask)

    # UNet: model outputs sigmoid probabilities in [0,1]; threshold at 0.5 matches reference app
    unet_mask = sam_pipeline.infer_mask(MODEL, rgb) & container_mask
    if not np.any(unet_mask):
        unet_mask = classical_mask.copy()

    # Keep only the largest connected component and fill holes
    final_mask = sam_pipeline.largest_component(unet_mask)
    final_mask = ndimage.binary_fill_holes(final_mask)

    props = measure.regionprops(final_mask.astype(int))[0]
    area_px = float(props.area)
    area_mm2 = area_px * (pixel_to_mm ** 2)
    equivalent_radius_mm = math.sqrt(area_mm2 / math.pi) if area_mm2 > 0 else 0.0
    diameter_mm = 2.0 * equivalent_radius_mm
    perimeter_mm = float(props.perimeter * pixel_to_mm)
    eccentricity = float(props.eccentricity)
    edge_roughness = float(props.perimeter / (2.0 * math.pi * max(props.equivalent_diameter / 2.0, 1.0)))

    corrected_gray = cv2.divide(gray, cv2.GaussianBlur(gray, (0, 0), 50), scale=255)
    filled = ndimage.binary_fill_holes(final_mask)
    local_t = threshold_local(corrected_gray, 51, offset=5)
    crack = (corrected_gray < local_t) & filled
    crack = morphology.remove_small_objects(crack, 15)
    crack_px = int(crack.sum())
    crack_area_mm2 = crack_px * (pixel_to_mm ** 2)
    crack_coverage_pct = 100.0 * crack_px / max(area_px, 1.0)
    crack_count = int(measure.label(crack).max())

    eq = exposure.equalize_adapthist(corrected_gray / 255.0)
    mej = meijering(eq, sigmas=range(1, 4), black_ridges=False)
    fra = frangi(eq, sigmas=range(1, 4), black_ridges=False)
    fra_thresh = filters.threshold_otsu(fra) if np.any(fra) else 0.0
    hyb = mej * (fra > 0.5 * fra_thresh)

    def skel(a):
        if not np.any(a):
            return np.zeros_like(a, bool)
        t = filters.threshold_otsu(a[a > 0]) if np.any(a > 0) else 0.1
        return morphology.skeletonize(morphology.remove_small_objects(a > 0.8 * t, 20))

    sk_f, sk_m, sk_h = skel(fra), skel(mej), skel(hyb)
    hyph_frangi_px, hyph_meijering_px, hyph_hybrid_px = map(int, [sk_f.sum(), sk_m.sum(), sk_h.sum()])

    overlay = _overlay_mask(rgb, final_mask)
    overlay_path = OVERLAY_DIR / f'{image_path.stem}_overlay.png'
    Image.fromarray(overlay).save(overlay_path)

    image_date = get_image_date(image_path)
    experiment_name = meta.get('experiment_name', infer_experiment_name(image_path)) if meta is not None else infer_experiment_name(image_path)
    experiment_date = meta.get('experiment_date', image_date) if meta is not None else image_date

    return {
        'image_path': image_path,
        'file': image_path.name,
        'experiment_name': experiment_name,
        'experiment_date': pd.to_datetime(experiment_date),
        'image_date': pd.to_datetime(image_date),
        'day_code': meta.get('day_code', '') if meta is not None else '',
        'rgb': rgb,
        'gray': gray,
        'pixel_to_mm': pixel_to_mm,
        'area_mm2': area_mm2,
        'diameter_mm': diameter_mm,
        'equivalent_radius_mm': equivalent_radius_mm,
        'perimeter_mm': perimeter_mm,
        'eccentricity': eccentricity,
        'edge_roughness': edge_roughness,
        'crack_px': crack_px,
        'crack_area_mm2': crack_area_mm2,
        'crack_coverage_pct': crack_coverage_pct,
        'crack_count': crack_count,
        'texture_entropy': float(shannon_entropy(gray[final_mask])) if np.any(final_mask) else 0.0,
        'hyph_frangi_px': hyph_frangi_px,
        'hyph_meijering_px': hyph_meijering_px,
        'hyph_hybrid_px': hyph_hybrid_px,
        'final_mask': final_mask,
        'classical_mask': classical_mask,
        'container_mask': container_mask,
        'overlay_path': overlay_path,
    }


In [ ]:
# Batch process all images and collect results
results = []
for _, row in tqdm(MANIFEST.iterrows(), total=len(MANIFEST), desc='pipelinesam'):
    t = trace_image(row['file_path'], meta=row)
    results.append(t)
    # Optionally display overlay
    # display(Image.open(t['overlay_path']))

DF = pd.DataFrame(results)
DF.to_csv(OUTPUTS_DIR / 'analysis_full.csv', index=False)
DF[['file', 'area_mm2', 'diameter_mm', 'crack_area_mm2', 'crack_count']].to_csv(OUTPUTS_DIR / 'analysis.csv', index=False)
DF.head()

In [ ]:
# Plot area and diameter over time
if not DF.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for exp, grp in DF.groupby('experiment_name'):
        grp = grp.sort_values('image_date')
        axes[0].plot(grp['image_date'], grp['area_mm2'], marker='o', label=exp)
        axes[1].plot(grp['image_date'], grp['diameter_mm'], marker='o', label=exp)
    axes[0].set(title='Sample Area Over Time', xlabel='Date', ylabel='Area (mm²)')
    axes[0].legend()
    axes[0].grid(alpha=0.2)
    axes[1].set(title='Sample Diameter Over Time', xlabel='Date', ylabel='Diameter (mm)')
    axes[1].legend()
    axes[1].grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print('No data to plot.')


In [ ]:
# Plot crack area and count over time
if not DF.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for exp, grp in DF.groupby('experiment_name'):
        grp = grp.sort_values('image_date')
        axes[0].plot(grp['image_date'], grp['crack_area_mm2'], marker='o', label=exp)
        axes[1].plot(grp['image_date'], grp['crack_count'], marker='o', label=exp)
    axes[0].set(title='Crack Area Over Time', xlabel='Date', ylabel='Crack Area (mm²)')
    axes[0].legend()
    axes[0].grid(alpha=0.2)
    axes[1].set(title='Crack Count Over Time', xlabel='Date', ylabel='Crack Count')
    axes[1].legend()
    axes[1].grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print('No data to plot.')


In [ ]:
# Save summary JSON for downstream use
import json
summary = DF[['file', 'area_mm2', 'diameter_mm', 'crack_area_mm2', 'crack_count']].to_dict(orient='records')
with open(OUTPUTS_DIR / 'analysis_full.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved summary to analysis_full.json')

In [ ]:
# Show overlays for a few samples
from IPython.display import display
for i, row in DF.head(3).iterrows():
    print(f"Sample: {row['file']}")
    display(Image.open(row['overlay_path']))

In [ ]:
# Show summary statistics for all samples
summary_stats = DF[['area_mm2', 'diameter_mm', 'crack_area_mm2', 'crack_count']].describe()
display(summary_stats)
summary_stats

In [ ]:
# Export overlays for all samples (optional)
for i, row in DF.iterrows():
    overlay = _overlay_mask(row['rgb'], row['final_mask'])
    overlay_path = OVERLAY_DIR / f"{row['file']}_overlay.png"
    Image.fromarray(overlay).save(overlay_path)
print('Exported overlays for all samples.')